# 02 — Preprocessing

Nettoyage, fusion des tables, encodage et feature engineering.
Resultat attendu : un DataFrame propre pret pour l entrainement.

## Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

## 1. Chargement des tables

In [2]:
app_train = pd.read_csv("../data/application_train.csv")
app_test  = pd.read_csv("../data/application_test.csv")
print("app_train :", app_train.shape)
print("app_test  :", app_test.shape)

app_train : (307511, 122)
app_test  : (48744, 121)


In [3]:
bureau          = pd.read_csv("../data/bureau.csv")
bureau_balance  = pd.read_csv("../data/bureau_balance.csv")
prev_app        = pd.read_csv("../data/previous_application.csv")
installments    = pd.read_csv("../data/installments_payments.csv")
pos_cash        = pd.read_csv("../data/POS_CASH_balance.csv")
cc_balance      = pd.read_csv("../data/credit_card_balance.csv")
print("Tables secondaires chargees")

Tables secondaires chargees


## 2. Nettoyage de application_train / test

### 2.1 Anomalie DAYS_EMPLOYED

Valeur 365243 = valeur codee signifiant "inconnu" (detectee en EDA). On la remplace par NaN et on cree un flag.

In [4]:
for df in [app_train, app_test]:
    df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
    df["DAYS_EMPLOYED"].replace({365243: np.nan}, inplace=True)

print("Anomalies train :", app_train["DAYS_EMPLOYED_ANOM"].sum())
print("Anomalies test  :", app_test["DAYS_EMPLOYED_ANOM"].sum())

Anomalies train : 55374
Anomalies test  : 9274


### 2.2 Suppression colonnes avec >60% de valeurs manquantes

Justification : trop peu d informations exploitables pour imputer de maniere fiable.

In [5]:
threshold = 0.6
missing_rate = app_train.isnull().mean()
cols_to_drop = missing_rate[missing_rate > threshold].index.tolist()
print(f"{len(cols_to_drop)} colonnes supprimees (>{int(threshold*100)}% manquants) :")
print(cols_to_drop)

app_train.drop(columns=cols_to_drop, inplace=True)
app_test.drop(columns=[c for c in cols_to_drop if c in app_test.columns], inplace=True)

17 colonnes supprimees (>60% manquants) :
['OWN_CAR_AGE', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'FLOORSMIN_AVG', 'LIVINGAPARTMENTS_AVG', 'NONLIVINGAPARTMENTS_AVG', 'YEARS_BUILD_MODE', 'COMMONAREA_MODE', 'FLOORSMIN_MODE', 'LIVINGAPARTMENTS_MODE', 'NONLIVINGAPARTMENTS_MODE', 'YEARS_BUILD_MEDI', 'COMMONAREA_MEDI', 'FLOORSMIN_MEDI', 'LIVINGAPARTMENTS_MEDI', 'NONLIVINGAPARTMENTS_MEDI', 'FONDKAPREMONT_MODE']


### 2.3 Imputation des valeurs manquantes restantes

Numeriques -> mediane (robuste aux outliers). Categorielles -> mode.

In [6]:
num_cols = app_train.select_dtypes(include=["number"]).columns
for col in num_cols:
    median = app_train[col].median()
    app_train[col].fillna(median, inplace=True)
    if col in app_test.columns:
        app_test[col].fillna(median, inplace=True)

cat_cols = app_train.select_dtypes(include=["object"]).columns
for col in cat_cols:
    mode = app_train[col].mode()[0]
    app_train[col].fillna(mode, inplace=True)
    if col in app_test.columns:
        app_test[col].fillna(mode, inplace=True)

print("Manquants restants train :", app_train.isnull().sum().sum())
print("Manquants restants test  :", app_test.isnull().sum().sum())

Manquants restants train : 0
Manquants restants test  : 0


## 3. Feature engineering — tables secondaires

On agreuge chaque table en une ligne par client (SK_ID_CURR) puis on la joint a app_train/test.

### 3.1 bureau.csv

In [7]:
bureau_agg = bureau.groupby("SK_ID_CURR").agg(
    BUREAU_COUNT           = ("SK_ID_BUREAU", "count"),
    BUREAU_CREDIT_SUM_MEAN = ("AMT_CREDIT_SUM", "mean"),
    BUREAU_CREDIT_SUM_MAX  = ("AMT_CREDIT_SUM", "max"),
    BUREAU_DAYS_CREDIT_MEAN= ("DAYS_CREDIT", "mean"),
).reset_index()
print("bureau_agg shape :", bureau_agg.shape)
bureau_agg.head()

bureau_agg shape : (305811, 5)


,SK_ID_CURR,BUREAU_COUNT,BUREAU_CREDIT_SUM_MEAN,BUREAU_CREDIT_SUM_MAX,BUREAU_DAYS_CREDIT_MEAN
0,100001,7,207623.571429,378000.0,-735.000000
1,100002,8,108131.945625,450000.0,-874.000000
2,100003,4,254350.125000,810000.0,-1400.750000
3,100004,2,94518.900000,94537.8,-867.000000
4,100005,3,219042.000000,568800.0,-190.666667


### 3.2 previous_application.csv

In [8]:
prev_agg = prev_app.groupby("SK_ID_CURR").agg(
    PREV_COUNT           = ("SK_ID_PREV", "count"),
    PREV_AMT_CREDIT_MEAN = ("AMT_CREDIT", "mean"),
    PREV_AMT_ANNUITY_MEAN= ("AMT_ANNUITY", "mean"),
).reset_index()
print("prev_agg shape :", prev_agg.shape)
prev_agg.head()

prev_agg shape : (338857, 4)


,SK_ID_CURR,PREV_COUNT,PREV_AMT_CREDIT_MEAN,PREV_AMT_ANNUITY_MEAN
0,100001,1,23787.00,3951.000
1,100002,1,179055.00,9251.775
2,100003,3,484191.00,56553.990
3,100004,1,20106.00,5357.250
4,100005,2,20076.75,4813.200


### 3.3 installments_payments.csv — features de retard de paiement

In [9]:
# Retard = difference entre montant du au montant paye
installments["PAYMENT_DIFF"] = installments["AMT_INSTALMENT"] - installments["AMT_PAYMENT"]
installments["DAYS_LATE"]    = installments["DAYS_ENTRY_PAYMENT"] - installments["DAYS_INSTALMENT"]

install_agg = installments.groupby("SK_ID_CURR").agg(
    INSTALL_COUNT             = ("SK_ID_PREV", "count"),
    INSTALL_PAYMENT_DIFF_MEAN = ("PAYMENT_DIFF", "mean"),
    INSTALL_DAYS_LATE_MEAN    = ("DAYS_LATE", "mean"),
    INSTALL_DAYS_LATE_MAX     = ("DAYS_LATE", "max"),
).reset_index()
print("install_agg shape :", install_agg.shape)
install_agg.head()

install_agg shape : (339587, 5)


,SK_ID_CURR,INSTALL_COUNT,INSTALL_PAYMENT_DIFF_MEAN,INSTALL_DAYS_LATE_MEAN,INSTALL_DAYS_LATE_MAX
0,100001,7,0.0,-7.285714,11.0
1,100002,19,0.0,-20.421053,-12.0
2,100003,25,0.0,-7.160000,-1.0
3,100004,3,0.0,-7.666667,-3.0
4,100005,9,0.0,-23.555556,1.0


### 3.4 bureau_balance.csv — Soldes mensuels des credits bureau

`bureau_balance` est liee a `bureau` par `SK_ID_BUREAU` (et non directement par `SK_ID_CURR`).
On agregue d'abord par `SK_ID_BUREAU`, puis on joint a `bureau` pour recuperer `SK_ID_CURR`,
et enfin on agregue par client.

In [10]:
# Etape 1 : agreger par SK_ID_BUREAU (nombre de mois suivis)
bb_agg_bureau = bureau_balance.groupby("SK_ID_BUREAU").agg(
    BB_MONTHS_COUNT = ("MONTHS_BALANCE", "count"),
    BB_MONTHS_MIN   = ("MONTHS_BALANCE", "min"),  # anciennete du suivi
).reset_index()

# Etape 2 : joindre a bureau pour recuperer SK_ID_CURR
bb_with_curr = bb_agg_bureau.merge(
    bureau[["SK_ID_BUREAU", "SK_ID_CURR"]],
    on="SK_ID_BUREAU", how="left"
)

# Etape 3 : agreger par client
bb_agg = bb_with_curr.groupby("SK_ID_CURR").agg(
    BB_TOTAL_MONTHS   = ("BB_MONTHS_COUNT", "sum"),
    BB_OLDEST_MONTH   = ("BB_MONTHS_MIN", "min"),
    BB_BUREAU_COUNT   = ("SK_ID_BUREAU", "count"),
).reset_index()
print("bb_agg shape :", bb_agg.shape)
bb_agg.head()

bb_agg shape : (134542, 4)


,SK_ID_CURR,BB_TOTAL_MONTHS,BB_OLDEST_MONTH,BB_BUREAU_COUNT
0,100001.0,172,-51,7
1,100002.0,110,-47,8
2,100005.0,21,-12,3
3,100010.0,72,-90,2
4,100013.0,230,-68,4


### 3.5 POS_CASH_balance.csv — Soldes POS et cash

Chaque ligne = un mois de suivi d'un ancien credit POS/cash.
On agregue par client pour extraire des indicateurs de comportement de paiement :
- Nombre total d'enregistrements (volume d'historique)
- Retards de paiement (`SK_DPD` = jours de retard)

In [11]:
pos_agg = pos_cash.groupby("SK_ID_CURR").agg(
    POS_COUNT        = ("SK_ID_PREV", "count"),
    POS_DPD_MEAN     = ("SK_DPD", "mean"),
    POS_DPD_MAX      = ("SK_DPD", "max"),
    POS_DPD_DEF_MEAN = ("SK_DPD_DEF", "mean"),
    POS_MONTHS_MIN   = ("MONTHS_BALANCE", "min"),
).reset_index()
print("pos_agg shape :", pos_agg.shape)
pos_agg.head()

pos_agg shape : (337252, 6)


,SK_ID_CURR,POS_COUNT,POS_DPD_MEAN,POS_DPD_MAX,POS_DPD_DEF_MEAN,POS_MONTHS_MIN
0,100001,9,0.777778,7,0.777778,-96
1,100002,19,0.000000,0,0.000000,-19
2,100003,28,0.000000,0,0.000000,-77
3,100004,4,0.000000,0,0.000000,-27
4,100005,11,0.000000,0,0.000000,-25


### 3.6 credit_card_balance.csv — Soldes cartes de credit

Chaque ligne = un mois de suivi d'une carte de credit.
On agregue par client pour capturer :
- Utilisation moyenne du credit (`AMT_BALANCE` vs `AMT_CREDIT_LIMIT_ACTUAL`)
- Nombre de tirages et retards de paiement

In [12]:
# Taux d'utilisation de la carte : solde / limite de credit
cc_balance["CC_UTILIZATION"] = cc_balance["AMT_BALANCE"] / cc_balance["AMT_CREDIT_LIMIT_ACTUAL"].replace(0, np.nan)

cc_agg = cc_balance.groupby("SK_ID_CURR").agg(
    CC_COUNT            = ("SK_ID_PREV", "count"),
    CC_BALANCE_MEAN     = ("AMT_BALANCE", "mean"),
    CC_UTILIZATION_MEAN = ("CC_UTILIZATION", "mean"),
    CC_DPD_MEAN         = ("SK_DPD", "mean"),
    CC_DPD_MAX          = ("SK_DPD", "max"),
    CC_DRAWINGS_MEAN    = ("AMT_DRAWINGS_CURRENT", "mean"),
).reset_index()
print("cc_agg shape :", cc_agg.shape)
cc_agg.head()

cc_agg shape : (103558, 7)


,SK_ID_CURR,CC_COUNT,CC_BALANCE_MEAN,CC_UTILIZATION_MEAN,CC_DPD_MEAN,CC_DPD_MAX,CC_DRAWINGS_MEAN
0,100006,6,0.000000,0.000000,0.000000,0,0.000000
1,100011,74,54482.111149,0.302678,0.000000,0,2432.432432
2,100013,96,18159.919219,0.115301,0.010417,1,5953.125000
3,100021,17,0.000000,0.000000,0.000000,0,0.000000
4,100023,8,0.000000,0.000000,0.000000,0,0.000000


## 4. Fusion de toutes les tables

In [13]:
def merge_all(df):
    df = df.merge(bureau_agg,  on="SK_ID_CURR", how="left")
    df = df.merge(prev_agg,    on="SK_ID_CURR", how="left")
    df = df.merge(install_agg, on="SK_ID_CURR", how="left")
    df = df.merge(bb_agg,      on="SK_ID_CURR", how="left")
    df = df.merge(pos_agg,     on="SK_ID_CURR", how="left")
    df = df.merge(cc_agg,      on="SK_ID_CURR", how="left")
    return df

app_train = merge_all(app_train)
app_test  = merge_all(app_test)

print("app_train apres fusion :", app_train.shape)
print("app_test  apres fusion :", app_test.shape)

app_train apres fusion : (307511, 131)
app_test  apres fusion : (48744, 130)


In [14]:
# Les clients sans historique dans les tables secondaires ont des NaN apres le left join
# On les impute a 0 (pas de credit = 0)
new_cols = (
    bureau_agg.columns.tolist()[1:] +
    prev_agg.columns.tolist()[1:] +
    install_agg.columns.tolist()[1:] +
    bb_agg.columns.tolist()[1:] +
    pos_agg.columns.tolist()[1:] +
    cc_agg.columns.tolist()[1:]
)
for col in new_cols:
    if col in app_train.columns:
        app_train[col].fillna(0, inplace=True)
    if col in app_test.columns:
        app_test[col].fillna(0, inplace=True)

print("Manquants apres fusion :", app_train.isnull().sum().sum())

Manquants apres fusion : 0


## 5. Encodage des variables categorielles

- **LabelEncoding** pour les variables binaires (<=2 categories)
- **OneHotEncoding** pour les variables nominales (>2 categories)

In [15]:
le = LabelEncoder()
le_count = 0

for col in app_train.select_dtypes("object").columns:
    if app_train[col].nunique() <= 2:
        le.fit(app_train[col])
        app_train[col] = le.transform(app_train[col])
        app_test[col]  = le.transform(app_test[col])
        le_count += 1

print(f"{le_count} colonnes label-encodees")

4 colonnes label-encodees


In [16]:
app_train = pd.get_dummies(app_train)
app_test  = pd.get_dummies(app_test)

# Alignement : garder uniquement les colonnes presentes dans les deux DataFrames
train_labels = app_train["TARGET"]
app_train, app_test = app_train.align(app_test, join="inner", axis=1)
app_train["TARGET"] = train_labels

print("app_train apres encodage :", app_train.shape)
print("app_test  apres encodage :", app_test.shape)

app_train apres encodage : (307511, 245)
app_test  apres encodage : (48744, 244)


## 6. Sauvegarde du dataset final

In [17]:
app_train.to_csv("../data/train_preprocessed.csv", index=False)
app_test.to_csv("../data/test_preprocessed.csv",   index=False)
print("Sauvegardes :")
print("  train_preprocessed.csv :", app_train.shape)
print("  test_preprocessed.csv  :", app_test.shape)

Sauvegardes :
  train_preprocessed.csv : (307511, 245)
  test_preprocessed.csv  : (48744, 244)


## Conclusion

- **6 tables secondaires fusionnees** : bureau + bureau_balance + previous_application + installments_payments + POS_CASH_balance + credit_card_balance
- Anomalie DAYS_EMPLOYED traitee (flag + remplacement NaN)
- Colonnes >60% manquants supprimees (17 colonnes)
- Imputation : mediane (numerique), mode (categoriel)
- Encodage : LabelEncoding (binaire) + OneHotEncoding (nominal) + alignement train/test
- Features ajoutees depuis les tables secondaires :
  - `bureau` : nombre de credits, montants moyens/max, anciennete
  - `bureau_balance` : nombre de mois suivis, anciennete du suivi
  - `previous_application` : nombre de demandes, montants moyens
  - `installments_payments` : retards de paiement (jours et montants)
  - `POS_CASH_balance` : retards DPD, anciennete du suivi POS
  - `credit_card_balance` : taux d'utilisation, solde moyen, tirages, retards DPD
- Dataset pret pour 03_modeling.ipynb